In [1]:
from collections import defaultdict, Counter

# from h_louvain import combined_modularity
from h_louvain import hLouvain
import hypernetx as hnx

from main import load_graph_and_communities
from data_loader import get_graph_nodes


# data = defaultdict(dict)
nodes = {}
hyperedges = {}
hyperdegree_distributions = {}
degree_distributions = {}
hyperedge_size_distributions = {}

for dataset in ['Algebra', 'Geometry', 'Restaurants-Rev', 'Music-Rev', 'Bars-Rev', 'contact-high-school', 'contact-primary-school']:
    graph, comm = load_graph_and_communities(dataset, tau=None, verbose=False)
    nodes[dataset] = len(get_graph_nodes(graph))
    hyperedges[dataset] = len(graph)

    node_hyperdegrees = Counter(node for hyperedge in graph.values() for node in hyperedge)
    hyperdegree_distributions[dataset] = node_hyperdegrees

    node_neighbors = defaultdict(set)
    for hyperedge in graph.values():
        for node in hyperedge:
            node_neighbors[node].update(n for n in hyperedge if n != node)
    degree_distributions[dataset] = {node: len(neighbors) for node, neighbors in node_neighbors.items()}

    hedge_sizes = Counter(len(hedge) for _, hedge in graph.items())
    hyperedge_size_distributions[dataset] = hedge_sizes

    # hnx_graph = hnx.Hypergraph(graph)
    # hL = hLouvain(hnx_graph, hmod_tau="infinity")
    # A, q2, alphas_out = hL.h_louvain_community(alphas=[1], change_mode="communities")
    # print(A)



    print(f"{dataset}: {len(node_hyperdegrees)} nodes, max degree={max(node_hyperdegrees.values()):d}")   



Algebra: 423 nodes, max degree=375
Geometry: 580 nodes, max degree=260
Restaurants-Rev: 565 nodes, max degree=59
Music-Rev: 1106 nodes, max degree=127
Bars-Rev: 1234 nodes, max degree=147
contact-high-school: 327 nodes, max degree=148
contact-primary-school: 242 nodes, max degree=261


In [ ]:
import matplotlib.pyplot as plt

for dataset, degrees in hyperdegree_distributions.items():
    counts = list(degrees.values())
    max_degree = max(counts) if counts else 0
    bins = range(0, max_degree + 2)
    plt.figure(figsize=(8, 4))
    plt.hist(counts, bins=bins, edgecolor='black', align='left')
    plt.yscale('log')
    plt.title(f'Degree distribution for {dataset}')
    plt.xlabel('Number of neighboring nodes')
    plt.ylabel('Node count (log scale)')
    plt.xticks(range(0, max_degree + 1, max(1, max_degree // 10)))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()